# 🏛️ IA para o Cidadão — Legislative Bill Analyzer

This notebook demonstrates an automated pipeline for analyzing Brazilian legislative bills (Projetos de Lei) using NLP and GPT-4o-mini.

**What it does:**
1. Extracts text from a PDF bill using `pdfplumber`
2. Summarizes the bill in accessible language
3. Classifies it by topic area
4. Generates a citizen-friendly explanation with practical examples

**Test document:** PL 2338/2023 — the Brazilian Artificial Intelligence Regulation Bill, one of the most significant tech policy proposals currently under debate in the Brazilian Congress.

> ⚠️ Running this notebook requires an OpenAI API key stored in a `.env` file.
> A sample output from a real run is shown at the end of this notebook.

## 1. Setup & Imports

In [ ]:
import sys
import os
import json

# Add project root to path so src modules are importable
project_root = os.path.abspath("..")
sys.path.append(project_root)

from src.helper_functions import analyze_bill
from src.prompts import (
    SUMMARY_PROMPT,
    CLASSIFICATION_PROMPT,
    SIMPLIFIED_EXPLANATION_PROMPT
)

print("✅ Setup complete.")

## 2. Load Extracted Bill Text

The PDF was previously processed by `extract_text.py`, which uses `pdfplumber` to extract raw text from any PDF bill downloaded from the Brazilian Chamber of Deputies website.

```bash
python -m src.extract_text data/PL2338_2023.pdf -o data/out_texts/
```

This produces a clean `.txt` file ready for the LLM pipeline.

In [ ]:
with open("../data/out_texts/PL2338_2023.txt", "r", encoding="utf-8") as f:
    texto_pl = f.read()

print(f"✅ Bill loaded: {len(texto_pl):,} characters")
print("\nFirst 500 characters:")
print(texto_pl[:500])

## 3. Run the Analysis Pipeline

The `analyze_bill()` function runs three sequential GPT-4o-mini calls:

| Step | Prompt | Output |
|---|---|---|
| Summary | `SUMMARY_PROMPT` | 3-paragraph accessible summary |
| Classification | `CLASSIFICATION_PROMPT` | Single topic category |
| Explanation | `SIMPLIFIED_EXPLANATION_PROMPT` | Citizen-friendly breakdown |

Each prompt is designed with a specific system persona and strict output instructions to minimize hallucination.

In [ ]:
prompts = {
    "summary": SUMMARY_PROMPT,
    "classification": CLASSIFICATION_PROMPT,
    "explanation": SIMPLIFIED_EXPLANATION_PROMPT
}

resultado = analyze_bill(texto_pl, prompts)

print("\n✅ Analysis complete.")

## 4. Results

In [ ]:
print("=" * 60)
print("SUMMARY")
print("=" * 60)
print(resultado["resumo"])

In [ ]:
print("=" * 60)
print("TOPIC CATEGORY")
print("=" * 60)
print(resultado["categoria"])

In [ ]:
print("=" * 60)
print("SIMPLIFIED EXPLANATION")
print("=" * 60)
print(resultado["explicacao_simplificada"])

## 5. Save Output to JSON

Results are saved as structured JSON for downstream use — for example, feeding into a Streamlit dashboard or a database.

In [ ]:
output_path = "../data/resultado_pl.json"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(resultado, f, ensure_ascii=False, indent=2)

print(f"✅ Results saved to {output_path}")

---
## 📋 Sample Output — Real API Run

The following shows actual output generated by running this pipeline on PL 2338/2023 (Brazilian AI Regulation Bill).
This allows you to evaluate the quality of results without needing an API key.

In [ ]:
sample_output = {
    "resumo": (
        "O Projeto de Lei nº 2338/2023 estabelece normas para o desenvolvimento e uso "
        "responsável de sistemas de inteligência artificial (IA) no Brasil, visando proteger "
        "os direitos fundamentais e garantir a segurança e confiabilidade desses sistemas. "
        "A proposta enfatiza a centralidade da pessoa humana, o respeito aos direitos humanos, "
        "a proteção do meio ambiente e a promoção da igualdade e não discriminação. Além disso, "
        "define princípios como transparência, auditabilidade e a necessidade de supervisão "
        "humana nas decisões tomadas por sistemas de IA.\n\n"
        "O projeto também assegura direitos às pessoas afetadas por sistemas de IA, como o "
        "direito à informação sobre interações com esses sistemas, o direito de contestar "
        "decisões automatizadas e o direito à não discriminação. Para sistemas considerados "
        "de alto risco, são exigidas avaliações de impacto e medidas de governança rigorosas, "
        "incluindo a documentação do funcionamento do sistema e a mitigação de vieses "
        "discriminatórios. A proposta prevê ainda a criação de uma autoridade competente "
        "para fiscalizar e regulamentar a aplicação da lei.\n\n"
        "Por fim, o projeto aborda a responsabilidade civil, estabelecendo que fornecedores "
        "e operadores de sistemas de IA devem reparar danos causados, com regras específicas "
        "para sistemas de alto risco. Também são previstas sanções administrativas para "
        "infrações, além de um ambiente regulatório experimental para fomentar a inovação em IA."
    ),
    "categoria": "Tecnologia e Inovação",
    "explicacao_simplificada": (
        "O Projeto de Lei de 2023 sobre IA no Brasil quer regular como essa tecnologia "
        "deve ser desenvolvida e usada, protegendo os direitos das pessoas.\n\n"
        "Em linguagem simples:\n"
        "- Você tem o direito de SABER quando está interagindo com uma IA\n"
        "- Você pode CONTESTAR decisões automatizadas (ex: crédito negado, vaga recusada)\n"
        "- Sistemas de ALTO RISCO (saúde, segurança, emprego) terão regras mais rígidas\n"
        "- Empresas são RESPONSÁVEIS por danos causados por seus sistemas de IA\n\n"
        "Exemplo prático: se um banco usa IA para negar seu empréstimo, "
        "você tem o direito de pedir uma revisão humana dessa decisão."
    )
}

print("CATEGORIA:", sample_output["categoria"])
print("\nRESUMO:")
print(sample_output["resumo"][:500], "...")
print("\nEXPLICAÇÃO SIMPLIFICADA:")
print(sample_output["explicacao_simplificada"])